# 04 — 3D Segmentation Inference

## Objective

This notebook applies the trained 2D U-Net to complete T2W MRI volumes.

Each axial slice is preprocessed and resized to the network input size,
segmented independently, and then restored to the original in-plane
resolution.

The resulting binary slices are reassembled into a 3D prostate mask
with the same array dimensions as the original MRI volume.

## 1. Environment Setup

The project root is added to the Python path so that the notebook can
access the reusable modules implemented in the `src` package.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II


## 2. Imports and Paths

This section imports the modules required to load an original PI-CAI
T2W volume, restore the trained U-Net checkpoint, and run full-volume
segmentation inference.

In [2]:
import torch
import numpy as np
import pandas as pd
import importlib
import SimpleITK as sitk

In [3]:
import src.data.paths
importlib.reload(src.data.paths)

import src.data.mapping
importlib.reload(src.data.mapping)

import src.data.loader
importlib.reload(src.data.loader)

import src.segmentation.model
importlib.reload(src.segmentation.model)

import src.segmentation.inference
importlib.reload(src.segmentation.inference)

<module 'src.segmentation.inference' from 'c:\\Users\\Luiz Trindade\\Documents\\penguin-adventure\\TCC_II\\src\\segmentation\\inference.py'>

In [4]:
from src.data.paths import (
    CHECKPOINTS_DIR,
)

from src.data.mapping import (
    get_case_image_paths,
    find_whole_gland_mask,
)

from src.data.loader import (
    load_medical_image,
)

from src.segmentation.model import (
    UNet,
)

from src.segmentation.inference import (
    predict_volume_masks,
    save_predicted_mask,
    predict_and_save_cases,
)

In [5]:
checkpoint_path = (
    CHECKPOINTS_DIR
    / "unet_whole_gland_grouped_v2_best.pth"
)

patient_id = "10001"
study_id = "1000001"

image_paths = get_case_image_paths(
    patient_id,
    study_id,
)

t2w_path = image_paths["t2w"]

print("T2W:", t2w_path)
print("Checkpoint:", checkpoint_path)

T2W: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\raw\picai\10001\10001_1000001_t2w.mha
Checkpoint: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\checkpoints\unet_whole_gland_grouped_v2_best.pth


## 3. Original T2W Volume

The original T2W MRI volume is loaded before segmentation inference.

The original array dimensions are preserved so that the predicted
whole-gland mask can later be reconstructed in the same image space.

In [6]:
t2w_image, t2w_volume = load_medical_image(
    t2w_path
)

print(
    "Volume shape:",
    t2w_volume.shape,
)

print(
    "Volume dtype:",
    t2w_volume.dtype,
)

print(
    "Image size:",
    t2w_image.GetSize(),
)

print(
    "Image spacing:",
    t2w_image.GetSpacing(),
)

Volume shape: (21, 640, 640)
Volume dtype: uint16
Image size: (640, 640, 21)
Image spacing: (0.30000001192093, 0.30000001192093, 3.6000000778884846)


## 4. Model Loading

The U-Net architecture is initialized and the best checkpoint selected
during validation is restored.

The model is placed in evaluation mode before segmentation inference.

In [7]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = UNet(
    in_channels=1,
    out_channels=1,
).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Device:", device)
print(
    "Checkpoint epoch:",
    checkpoint["epoch"],
)

Device: cuda
Checkpoint epoch: 5


## 5. Full-Volume Segmentation Inference

The trained 2D U-Net is applied independently to every axial slice of
the original T2W volume.

Each slice is normalized and resized to the network input resolution of
256 × 256 pixels. After inference, the binary prediction is resized back
to the original in-plane resolution using nearest-neighbor interpolation.

Unlike the training-data preparation procedure, all axial slices are
processed during inference. Ground-truth masks are therefore not used
to determine which slices should be segmented.

In [8]:
predicted_volume = predict_volume_masks(
    model=model,
    volume=t2w_volume,
    device=device,
    threshold=0.5,
)

print(
    "Original volume shape:",
    t2w_volume.shape,
)

print(
    "Predicted mask shape:",
    predicted_volume.shape,
)

print(
    "Predicted mask dtype:",
    predicted_volume.dtype,
)

print(
    "Predicted mask values:",
    sorted(set(predicted_volume.ravel())),
)

print(
    "Predicted positive voxels:",
    int(predicted_volume.sum()),
)

Original volume shape: (21, 640, 640)
Predicted mask shape: (21, 640, 640)
Predicted mask dtype: uint8
Predicted mask values: [np.uint8(0), np.uint8(1)]
Predicted positive voxels: 279860


## 6. 3D Segmentation Validation

For validation of the reconstruction procedure, the predicted 3D mask
is compared with the reference whole-gland annotation available for the
selected validation case.

Dice and Intersection over Union (IoU) are calculated over the complete
3D volume rather than independently for each axial slice.

The reference annotation is used only to evaluate the segmentation and
does not participate in the inference procedure.

In [9]:
mask_path = find_whole_gland_mask(
    patient_id,
    study_id,
)

mask_image, ground_truth_volume = (
    load_medical_image(mask_path)
)

ground_truth_positive_slices = [
    index
    for index in range(
        ground_truth_volume.shape[0]
    )
    if ground_truth_volume[index].sum() > 0
]

print(
    "Ground-truth shape:",
    ground_truth_volume.shape,
)

print(
    "Ground-truth positive slices:",
    ground_truth_positive_slices,
)

print(
    "Number of ground-truth positive slices:",
    len(ground_truth_positive_slices),
)

Ground-truth shape: (21, 640, 640)
Ground-truth positive slices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Number of ground-truth positive slices: 17


In [10]:
ground_truth_binary = (
    ground_truth_volume > 0
).astype("uint8")

intersection = (
    predicted_volume
    * ground_truth_binary
).sum()

predicted_voxels = (
    predicted_volume.sum()
)

ground_truth_voxels = (
    ground_truth_binary.sum()
)

volume_dice = (
    2.0 * intersection
    / (
        predicted_voxels
        + ground_truth_voxels
    )
)

union = (
    (
        predicted_volume
        + ground_truth_binary
    ) > 0
).sum()

volume_iou = (
    intersection
    / union
)

print(
    "3D Dice:",
    float(volume_dice),
)

print(
    "3D IoU:",
    float(volume_iou),
)

print(
    "Predicted voxels:",
    int(predicted_voxels),
)

print(
    "Ground-truth voxels:",
    int(ground_truth_voxels),
)

3D Dice: 0.9497209112646933
3D IoU: 0.9042557558756115
Predicted voxels: 279860
Ground-truth voxels: 276773


In [16]:
patient_id = "11050"
study_id = "1001070"

pred_path = (
    PROJECT_ROOT
    / "src/data/processed/whole_gland_predictions_grouped_v2"
    / f"{patient_id}_{study_id}_whole_gland_pred.mha"
)

ref_path = find_whole_gland_mask(patient_id, study_id)

pred_img = sitk.ReadImage(str(pred_path))
ref_img = sitk.ReadImage(str(ref_path))

ref_aligned = sitk.Resample(
    ref_img,
    pred_img,
    sitk.Transform(3, sitk.sitkIdentity),
    sitk.sitkNearestNeighbor,
    0,
    ref_img.GetPixelID()
)

pred = sitk.GetArrayFromImage(pred_img) > 0
ref = sitk.GetArrayFromImage(ref_aligned) > 0

def mostrar_regiao(nome, mascara):
    coordenadas = np.argwhere(mascara)
    if len(coordenadas) == 0:
        print(nome, "- máscara vazia")
        return

    print(f"\n{nome}")
    print("Voxels:", len(coordenadas))
    print("Início (z,y,x):", coordenadas.min(axis=0))
    print("Fim (z,y,x):", coordenadas.max(axis=0))

mostrar_regiao("Predição", pred)
mostrar_regiao("Referência alinhada", ref)

print("\nGeometria original:")
print("Predição - Origin:", pred_img.GetOrigin())
print("Referência - Origin:", ref_img.GetOrigin())
print("Predição - Spacing:", pred_img.GetSpacing())
print("Referência - Spacing:", ref_img.GetSpacing())
print("Predição - Direction:", pred_img.GetDirection())
print("Referência - Direction:", ref_img.GetDirection())

print("\nInterseção:", np.count_nonzero(pred & ref))


Predição
Voxels: 123828
Início (z,y,x): [  5 283 230]
Fim (z,y,x): [ 16 404 414]

Referência alinhada
Voxels: 184476
Início (z,y,x): [22  4 85]
Fim (z,y,x): [ 29 272 392]

Geometria original:
Predição - Origin: (-92.279447321886, -86.590177859754, -25.540836738613)
Referência - Origin: (-92.27944946289062, -86.59017944335938, -25.540836334228516)
Predição - Spacing: (0.28125, 0.28125, 3.4500000138306155)
Referência - Spacing: (0.28125, 0.28125, 3.450000047683716)
Predição - Direction: (0.999258266441154, 1.667079998352028e-10, 0.038508660701451806, -0.012227581151884282, 0.9482488703359125, 0.3172925561147625, -0.03651579395540542, -0.3175280773516975, 0.9475455223286852)
Referência - Direction: (0.9992583534801338, -1.3878505879726532e-05, 0.03850639999442363, -0.012213701932352902, 0.9482489644768445, 0.3172927874386403, -0.03651805697568701, -0.31752779591081376, 0.9475455367415792)

Interseção: 0


The reconstructed 3D segmentation is evaluated above using the Dice and IoU metrics.

Report the **newly executed values** for this grouped-v2 validation case; do not
reuse metrics from the historical checkpoint.


## 7. Physical-Space Reconstruction

The predicted NumPy volume contains the segmentation in the original
array dimensions, but it does not independently store medical-image
spatial metadata.

A SimpleITK image is therefore created from the predicted volume and the
size, spacing, origin, and direction information of the original T2W MRI
is preserved.

This ensures that the predicted segmentation occupies the same physical
image space as the source MRI.

In [11]:
predicted_mask_image = sitk.GetImageFromArray(
    predicted_volume.astype("uint8")
)

predicted_mask_image.CopyInformation(
    t2w_image
)

print(
    "Predicted mask size:",
    predicted_mask_image.GetSize(),
)

print(
    "Predicted mask spacing:",
    predicted_mask_image.GetSpacing(),
)

print(
    "Predicted mask origin:",
    predicted_mask_image.GetOrigin(),
)

print(
    "Predicted mask direction:",
    predicted_mask_image.GetDirection(),
)

Predicted mask size: (640, 640, 21)
Predicted mask spacing: (0.30000001192093, 0.30000001192093, 3.6000000778884846)
Predicted mask origin: (-119.80861242576, -65.872387498812, -38.779758120899)
Predicted mask direction: (0.9996148108062515, -1.6670999975453016e-11, -0.02775301815627119, 0.00043710598429775216, 0.999875963366245, 0.01574378673580096, 0.027749575765060076, -0.015749853409600077, 0.9994908219500744)


## 8. Saving the Predicted 3D Mask

The reconstructed whole-gland segmentation is saved as an MHA medical
image.

The saved mask preserves the spatial metadata of the original T2W MRI,
allowing it to be used together with the source image in subsequent
radiomic feature extraction.

In [12]:
prediction_dir = (
    PROJECT_ROOT
    / "results"
    / "segmentation"
    / "predicted_masks_grouped_v2_check"
)

prediction_dir.mkdir(
    parents=True,
    exist_ok=True,
)

predicted_mask_path = (
    prediction_dir
    / f"{patient_id}_{study_id}_whole_gland_pred.mha"
)

sitk.WriteImage(
    predicted_mask_image,
    str(predicted_mask_path),
)

print(
    "Predicted mask saved to:",
    predicted_mask_path,
)

Predicted mask saved to: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\segmentation\predicted_masks_grouped_v2_check\10001_1000001_whole_gland_pred.mha


### Saved Mask Verification

The saved segmentation is reloaded to verify that its binary content and
spatial metadata were preserved during serialization.

In [13]:
reloaded_mask_image, reloaded_mask_volume = (
    load_medical_image(
        predicted_mask_path
    )
)

print(
    "Reloaded shape:",
    reloaded_mask_volume.shape,
)

print(
    "Reloaded dtype:",
    reloaded_mask_volume.dtype,
)

print(
    "Reloaded values:",
    sorted(
        set(
            reloaded_mask_volume.ravel()
        )
    ),
)

print(
    "Same size:",
    reloaded_mask_image.GetSize()
    == t2w_image.GetSize(),
)

print(
    "Same spacing:",
    reloaded_mask_image.GetSpacing()
    == t2w_image.GetSpacing(),
)

print(
    "Same origin:",
    reloaded_mask_image.GetOrigin()
    == t2w_image.GetOrigin(),
)

print(
    "Same direction:",
    reloaded_mask_image.GetDirection()
    == t2w_image.GetDirection(),
)

Reloaded shape: (21, 640, 640)
Reloaded dtype: uint8
Reloaded values: [np.uint8(0), np.uint8(1)]
Same size: True
Same spacing: True
Same origin: True
Same direction: True


## 9. Inference Summary

The full-volume inference workflow predicts a binary 3D whole-gland segmentation
without using the reference annotation to select slices.

The reconstructed prediction is evaluated on the selected **validation** case,
using the 3D Dice and IoU calculated in section 6. Record those new values
only after executing the notebook.

After saving and reloading the prediction, the size, spacing, origin, direction,
and binary representation can be checked against the source T2W volume.
The saved mask is written to a separate verification directory so the historical
masks and the existing 1,500 grouped-v2 masks are not overwritten.


## 10. Compare with an existing grouped-v2 prediction

The project already contains a full set of masks in
`src/data/processed/whole_gland_predictions_grouped_v2/`.
This optional read-only check compares the reconstructed mask for this case
with the corresponding pre-existing mask. Agreement for one case is useful
evidence of consistency, but does not verify all 1,500 predictions.


In [14]:
existing_mask_path = (
    PROJECT_ROOT
    / "src" / "data" / "processed"
    / "whole_gland_predictions_grouped_v2"
    / f"{patient_id}_{study_id}_whole_gland_pred.mha"
)

print("Existing grouped-v2 mask:", existing_mask_path)
if not existing_mask_path.is_file():
    print("Mask not found; comparison skipped.")
else:
    existing_image, existing_volume = load_medical_image(existing_mask_path)
    same_shape = existing_volume.shape == predicted_volume.shape
    print("Same shape:", same_shape)
    if same_shape:
        print("Same binary voxels:", bool(np.array_equal(existing_volume > 0, predicted_volume > 0)))
        print("Different voxels:", int(np.count_nonzero((existing_volume > 0) != (predicted_volume > 0))))
    print("Same spacing:", existing_image.GetSpacing() == t2w_image.GetSpacing())
    print("Same origin:", existing_image.GetOrigin() == t2w_image.GetOrigin())
    print("Same direction:", existing_image.GetDirection() == t2w_image.GetDirection())


Existing grouped-v2 mask: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\whole_gland_predictions_grouped_v2\10001_1000001_whole_gland_pred.mha
Same shape: True
Same binary voxels: True
Different voxels: 0
Same spacing: True
Same origin: True
Same direction: True


## 11. Avaliação 3D — todos os exames do teste grouped_v2

Esta etapa **não executa novamente a U-Net e não modifica as máscaras armazenadas**.
Ela compara as previsões já salvas de `whole_gland_predictions_grouped_v2`
com as máscaras anatômicas de referência para os **225 exames de teste**, conforme
`patient_splits_grouped_v2.csv`.

Antes do cálculo, verifica se a máscara predita mantém a geometria da imagem T2W.
Se a geometria da referência diferir, alinha **somente a referência** à máscara
predita com interpolação por vizinho mais próximo e registra o procedimento.
Casos com problemas não entram nas médias e ficam identificados no CSV.

**Requisito de execução:** executar as células de configuração e importação até
`## 2. Imports and Paths`; esta etapa independe da inferência do caso individual.


In [15]:
# Avaliação 3D do conjunto TESTE grouped_v2 (somente leitura dos volumes).
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk

splits_path = PROJECT_ROOT / "src/data/processed/splits/patient_splits_grouped_v2.csv"
predictions_dir = PROJECT_ROOT / "src/data/processed/whole_gland_predictions_grouped_v2"
results_dir = PROJECT_ROOT / "results/segmentation"

assert splits_path.is_file(), f"CSV não encontrado: {splits_path}"
assert predictions_dir.is_dir(), f"Máscaras não encontradas: {predictions_dir}"
splits_3d = pd.read_csv(splits_path, dtype={"patient_id": str, "study_id": str})
splits_3d["split"] = splits_3d["split"].str.strip().str.lower()
assert not splits_3d.duplicated(["patient_id", "study_id"]).any(), "Exames duplicados no CSV"
assert (splits_3d.groupby("patient_id")["split"].nunique() <= 1).all(), "Vazamento de pacientes no CSV"
test_cases_3d = splits_3d.loc[splits_3d["split"] == "test", ["patient_id", "study_id"]]
assert len(test_cases_3d) == 225, f"Esperados 225 exames de teste; encontrados {len(test_cases_3d)}"

# Tolerância apenas para arredondamentos em metadados espaciais.
def same_grid_3d(a, b):
    return (
        a.GetSize() == b.GetSize()
        and np.allclose(a.GetSpacing(), b.GetSpacing(), atol=1e-5, rtol=0)
        and np.allclose(a.GetOrigin(), b.GetOrigin(), atol=1e-4, rtol=0)
        and np.allclose(a.GetDirection(), b.GetDirection(), atol=1e-6, rtol=0)
    )

records_3d = []
for case in test_cases_3d.itertuples(index=False):
    patient, study = case.patient_id, case.study_id
    row = {"patient_id": patient, "study_id": study, "split": "test",
           "status": "ok", "reference_resampled": False, "error": ""}
    try:
        pred_path = predictions_dir / f"{patient}_{study}_whole_gland_pred.mha"
        if not pred_path.is_file():
            raise FileNotFoundError(f"Máscara predita ausente: {pred_path}")
        pred_image = sitk.ReadImage(str(pred_path))
        t2w_path = get_case_image_paths(patient, study)["t2w"]
        t2w_image = sitk.ReadImage(str(t2w_path))
        if not same_grid_3d(pred_image, t2w_image):
            raise ValueError("Geometria da predição não corresponde à T2W")

        ref_path = find_whole_gland_mask(patient, study)
        ref_image = sitk.ReadImage(str(ref_path))
        if not same_grid_3d(ref_image, pred_image):
            ref_image_aligned = sitk.Resample(
                ref_image, pred_image, sitk.Transform(3, sitk.sitkIdentity),
                sitk.sitkNearestNeighbor, 0, ref_image.GetPixelID()
            )
            row["reference_resampled"] = True
        else:
            ref_image_aligned = ref_image

        pred = sitk.GetArrayFromImage(pred_image) > 0
        ref = sitk.GetArrayFromImage(ref_image_aligned) > 0
        ref_original_has_voxels = bool(sitk.GetArrayViewFromImage(ref_image).any())
        if ref_original_has_voxels and not ref.any():
            raise ValueError("Referência ficou vazia após alinhamento: revisar geometria")
        assert pred.shape == ref.shape

        intersection = int(np.count_nonzero(pred & ref))
        n_pred = int(np.count_nonzero(pred))
        n_ref = int(np.count_nonzero(ref))
        denominator = n_pred + n_ref
        union = denominator - intersection
        row.update({
            "dice_3d": 2 * intersection / denominator if denominator else 1.0,
            "iou_3d": intersection / union if union else 1.0,
            "voxels_predicted": n_pred,
            "voxels_reference": n_ref,
            "voxels_intersection": intersection,
        })
    except Exception as exc:
        row["status"] = "error"
        row["error"] = f"{type(exc).__name__}: {exc}"
    records_3d.append(row)

results_3d = pd.DataFrame(records_3d)
valid_3d = results_3d.loc[results_3d["status"] == "ok"]
summary_3d = valid_3d[["dice_3d", "iou_3d"]].agg(
    ["count", "mean", "median", "std", "min", "max"]
).T

results_dir.mkdir(parents=True, exist_ok=True)
metrics_path = results_dir / "grouped_v2_test_3d_metrics.csv"
summary_path = results_dir / "grouped_v2_test_3d_summary.csv"
results_3d.to_csv(metrics_path, index=False)
summary_3d.to_csv(summary_path, index_label="metric")

print(f"Exames de teste: {len(test_cases_3d)}")
print(f"Avaliados: {len(valid_3d)} | Erros: {(results_3d['status'] == 'error').sum()}")
print(f"Referências reamostradas: {int(valid_3d['reference_resampled'].sum())}")
print("\nResumo 3D:")
print(summary_3d.round(4).to_string())
if len(valid_3d) != len(test_cases_3d):
    print("\nATENÇÃO: avaliação incompleta; verifique os casos com erro antes de relatar médias finais.")
    print(results_3d.loc[results_3d["status"] != "ok", ["patient_id", "study_id", "error"]].to_string(index=False))
print("\nCSV individual:", metrics_path)
print("CSV resumo:", summary_path)


Exames de teste: 225
Avaliados: 225 | Erros: 0
Referências reamostradas: 45

Resumo 3D:
         count    mean  median     std  min     max
dice_3d  225.0  0.9045  0.9216  0.0763  0.0  0.9613
iou_3d   225.0  0.8317  0.8546  0.0900  0.0  0.9255

CSV individual: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\segmentation\grouped_v2_test_3d_metrics.csv
CSV resumo: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\segmentation\grouped_v2_test_3d_summary.csv
